In [2]:
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET
import zipfile

# ============================================================
# PT Step — Add GTFS route colours to SUMO PT vehicles
# ============================================================

# ----------------------
# CONFIG
# ----------------------
PT_DIR = Path("./")  # Folder where gtfs2pt outputs are located
GTFS_ZIP_OR_DIR = Path("./input/ca_la_rochelle-aggregated-gtfs.zip")  # GTFS as .zip or folder

# Optional: if you know the exact file, set it here (recommended)
# VEHICLES_XML_HINT = PT_DIR / "gtfs_pt_vehicles.add.xml"
VEHICLES_XML_HINT = None


# ----------------------
# Helpers
# ----------------------
def hex_to_rgb_str(hex_color: str):
    """'#RRGGBB' or 'RRGGBB' -> 'r,g,b' (0-255). Return None if invalid."""
    if not isinstance(hex_color, str):
        return None
    h = hex_color.strip().lstrip("#")
    if len(h) != 6:
        return None
    try:
        r = int(h[0:2], 16)
        g = int(h[2:4], 16)
        b = int(h[4:6], 16)
    except ValueError:
        return None
    return f"{r},{g},{b}"


def read_gtfs_table(gtfs_path: Path, filename: str) -> pd.DataFrame:
    """Read a GTFS CSV from either a zip file or a folder."""
    if gtfs_path.is_file() and gtfs_path.suffix.lower() == ".zip":
        with zipfile.ZipFile(gtfs_path, "r") as z:
            if filename not in z.namelist():
                raise FileNotFoundError(f"'{filename}' not found inside GTFS zip: {gtfs_path}")
            with z.open(filename) as f:
                return pd.read_csv(f, dtype=str)
    else:
        file_path = gtfs_path / filename
        if not file_path.exists():
            raise FileNotFoundError(f"GTFS file not found: {file_path.resolve()}")
        return pd.read_csv(file_path, dtype=str)


def strip_ns(tag: str) -> str:
    """Remove XML namespace if present."""
    return tag.split("}", 1)[-1] if "}" in tag else tag


def detect_vehicles_xml(pt_dir: Path) -> Path:
    """
    Try to locate the SUMO PT vehicles file produced by gtfs2pt.
    Strategy:
    1) Prefer a file name containing 'vehicles' and '.add.xml'
    2) Otherwise scan XML files and look for <vehicle ... line="...">
    """
    # 1) Filename-based heuristic (fast)
    candidates = sorted(pt_dir.rglob("*vehicles*.add.xml"))
    if candidates:
        return candidates[0]

    # 2) Content-based fallback (slower)
    for p in sorted(pt_dir.rglob("*.xml")):
        try:
            txt = p.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        if "<vehicle" in txt and ' line="' in txt:
            return p

    raise FileNotFoundError(
        "Could not find an XML file containing <vehicle ... line=\"...\"> under PT_DIR."
    )


# ----------------------
# 1) Read GTFS routes + trips
# ----------------------
routes = read_gtfs_table(GTFS_ZIP_OR_DIR, "routes.txt")
trips = read_gtfs_table(GTFS_ZIP_OR_DIR, "trips.txt")

if "route_id" not in routes.columns:
    raise ValueError("routes.txt must contain column: route_id")

if "route_color" not in routes.columns:
    raise ValueError(
        "routes.txt does not contain route_color. "
        "In that case you need a manual line->colour mapping (or another source)."
    )

if not {"trip_id", "route_id"}.issubset(trips.columns):
    raise ValueError("trips.txt must contain columns: trip_id and route_id")

routes = routes.copy()
if "route_short_name" not in routes.columns:
    routes["route_short_name"] = pd.NA

routes["rgb"] = routes["route_color"].apply(hex_to_rgb_str)

route_info = (
    routes[["route_id", "route_short_name", "route_color", "rgb"]]
    .drop_duplicates("route_id")
    .set_index("route_id")
    .to_dict(orient="index")
)

trip_to_route = (
    trips[["trip_id", "route_id"]]
    .drop_duplicates("trip_id")
    .set_index("trip_id")["route_id"]
    .to_dict()
)

# ----------------------
# 2) Locate SUMO vehicles file (gtfs_pt_vehicles.add.xml)
# ----------------------
if VEHICLES_XML_HINT is not None:
    vehicles_xml = VEHICLES_XML_HINT
    if not vehicles_xml.exists():
        raise FileNotFoundError(f"Vehicles XML not found: {vehicles_xml.resolve()}")
else:
    vehicles_xml = detect_vehicles_xml(PT_DIR)

print(f"[OK] Vehicles file detected: {vehicles_xml}")

# ----------------------
# 3) Patch colours
# ----------------------
tree = ET.parse(vehicles_xml)
root = tree.getroot()

total = 0
colored = 0
missing_trip = 0
missing_route = 0
missing_color = 0

for el in root.iter():
    if strip_ns(el.tag) != "vehicle":
        continue

    total += 1
    line_value = el.get("line")  # in gtfs2pt outputs, this is often a GTFS trip_id (but may differ)
    if not line_value:
        missing_trip += 1
        continue

    # First attempt: treat 'line' as trip_id -> route_id
    route_id = trip_to_route.get(line_value)

    # Fallback: sometimes 'line' may already be a route_id or a line identifier
    if route_id is None:
        if line_value in route_info:
            route_id = line_value
        else:
            missing_route += 1
            continue

    info = route_info.get(route_id, {})
    rgb = info.get("rgb")
    if not rgb:
        missing_color += 1
        continue

    el.set("color", rgb)
    colored += 1

# Write output
suffix = "_colored.add.xml"
out_path = vehicles_xml.with_name(vehicles_xml.name.replace(".add.xml", suffix))
tree.write(out_path, encoding="utf-8", xml_declaration=True)

print("\n========== SUMMARY ==========")
print(f"[INFO] Total vehicles: {total}")
print(f"[INFO] Coloured vehicles: {colored}")
print(f"[INFO] Missing/empty 'line' attribute: {missing_trip}")
print(f"[INFO] Unknown line->route mapping: {missing_route}")
print(f"[INFO] Missing/invalid route colours: {missing_color}")
print(f"[OK] Output written: {out_path}")

# ----------------------
# 4) Optional: export a small mapping CSV for debugging
# ----------------------
rows = []
for trip_id, route_id in trip_to_route.items():
    info = route_info.get(route_id, {})
    rows.append(
        {
            "trip_id": trip_id,
            "route_id": route_id,
            "route_short_name": info.get("route_short_name"),
            "route_color_hex": info.get("route_color"),
            "rgb": info.get("rgb"),
        }
    )

mapping_df = pd.DataFrame(rows)
mapping_csv = PT_DIR / "trip_to_route_color.csv"
mapping_df.to_csv(mapping_csv, index=False)
print(f"[OK] Debug mapping CSV written: {mapping_csv}")

[OK] Vehicles file detected: gtfs_pt_vehicles.add.xml

========== SUMMARY ==========
[INFO] Total vehicles: 1595
[INFO] Coloured vehicles: 1595
[INFO] Missing/empty 'line' attribute: 0
[INFO] Unknown line->route mapping: 0
[INFO] Missing/invalid route colours: 0
[OK] Output written: gtfs_pt_vehicles_colored.add.xml
[OK] Debug mapping CSV written: trip_to_route_color.csv
